# BetaTour run v23: yeniden başlatma işaretlemesi (keşif koşusu)

**Fikir.** LKH-3 ilk denemeden sonraki her denemeye en iyi turun büyük bölümünü koruyarak başlar. Bir tur kenarı yalnızca aday dosyasında **alfa değeri 0** ise korunur (`ChooseInitialTour.c`, C durumu); korunmayan yerlerde tur rastgele bozulur ve o denemenin araması yalnızca bu yerlerden başlayabilir (`LinKernighan.c`, kısıtlı arama). Alfa = 0 işaretleri bu yüzden her denemede turun neresinin açılacağını belirler. LKH bu işaretleri en küçük 1-ağaca göre koyar; run v21'in düzeltilmiş dosyaları da aynı kuralı kullandı.

Bu koşu, korunacak kenarları tur topluluğunun seçmesini dener:

- **Sayım işaretlemesi:** altı ilk deneme turunun en az beşinde bulunan kenarlar korunur.
- **Model işaretlemesi:** modelin tahmin olasılığı bir eşiğin üstünde olan kenarlar korunur. Eşik tohum başına seçilir: tohumun ilk turunda sayım işaretlemesiyle aynı sayıda kenar korunur. İki işaretleme böylece yalnızca *hangi* kenarları seçtiğinde ayrılır.

**Elimizdeki kanıt zayıftır.** Geliştirme ortamında 10.000 düğümlü üç örnekte LKH'nin kuralı ilk turun kenarlarının %24–25'ini açıkta bıraktı; turun, 150 saniyelik ayrı bir koşunun bulduğu daha iyi turda bulunmayan kenarlarının %59–62'si bu açıkta kalanlar arasındaydı. Sayım işaretlemesi kenarların %16–19'unu açıkta bırakıp bu kenarların %78–83'ünü, model işaretlemesi aynı payda %72–81'ini yakaladı. Aynı ortamda LKH ile yapılan küçük ön deneme (8 tek tohumlu koşu, 60 s) ise belirgin bir kazanç göstermedi (model eşikli işaretleme −%0,009; 5 koşuda kısa, 3 koşuda uzun; gürültü düzeyinde). Bu nedenle koşu **keşif amaçlıdır**: bir işaretleme daha iyi görünürse, makalede yer almadan önce ayrı bir doğrulama koşusunda sınanır. Sonuç hangi yönde çıkarsa çıksın kaydedilir.

**Tasarım.** Daha önce hiçbir koşuda kullanılmamış 12 örnek (10.000 düğüm; 6 uniform, 6 kümeli), bütçe 120 s, altı tohum (101–106), altı paralel LKH-3.0.14 süreci. Altı kol (iki liste × iki işaretleme ve BetaTour listesinde sayım):

| Kol | Aday listesi | Alfa = 0 işaretleri |
|---|---|---|
| `pop_full` | POPMUSIC-5, kesintisiz | LKH'nin kendisi (dosya yok) |
| `pop_native` | aynı tohumun POPMUSIC-5 listesi, yeniden başlatma | LKH'nin 1-ağacı |
| `pop_model` | aynı liste, yeniden başlatma | model işaretlemesi |
| `beta_tree` | BetaTour karma listesi (model ilk 3 + POPMUSIC) | LKH'nin 1-ağacı (run v21'deki kol) |
| `beta_model` | aynı BetaTour listesi | model işaretlemesi |
| `beta_count` | aynı BetaTour listesi | sayım işaretlemesi |

Bütün dosyalarda baba (Dad) alanı aynı tohumun kendi dosyasından gelir. Zaman hesabı run v21 ile aynıdır: yeniden başlatılan kolların bütçesinden ortak ilk aşamanın ve dosyanın hazırlık süresi (model, liste, işaretleme) düşülür.

**Önceden yazılan üç ana okuma** (`experiments/PROTOCOL_v23.md`, koşudan önce yazıldı; bütçe sonunda, altı tohumun eşli ortalaması):

- **M1:** `beta_model` / `beta_tree`. Model işaretlemesi BetaTour'u iyileştiriyor mu?
- **M2:** `beta_count` / `beta_tree`. Sayım işaretlemesi iyileştiriyor mu?
- **M3:** `pop_model` / `pop_native`. İşaretleme LKH'nin kendi listelerinde de işe yarıyor mu?

Karar kuralı: bir işaretleme, örnek düzeyinde kesin iki yönlü işaret testinde p < 0,05 ile daha kısa çıkarsa (eşitlik yoksa 12 örneğin en az 10'unda) onun için doğrulama koşusu hazırlanır; değilse fikir bu boyutta desteklenmemiş sayılır.

**Çalıştırma.** Colab'da **A100 veya L4 GPU** ve **Yüksek RAM** (12 vCPU) seçin, sonra *Çalışma zamanı → Tümünü çalıştır*. Drive izni istendiğinde onaylayın ve sekmeyi açık tutun. Bağlantı koparsa aynı ortam türüyle yeniden *Tümünü çalıştır* deyin: biten örnekler Drive'daki `MyDrive/BetaTour/v23_marking12` klasöründen okunur ve atlanır; yarım kalan örnek baştan koşar.

**Tahmini süre.**

| Aşama | Süre |
|---|---|
| Kurulum, depo, LKH derlemesi | ≈ 3–5 dk |
| Isınma ve kodlama öz sınaması | ≈ 1 dk |
| Örnek başına: ortak ilk aşama ≈ 15 s, kesintisiz kol ≈ 2 dk, beş yeniden başlatılan kol ≈ 9 dk | ≈ 11,5 dk |
| 12 örnek | **≈ 2,3 saat** |
| Çözümleme ve ZIP | ≈ 1 dk |

Daha kısa oturumlar için ilk hücrede `MAX_NEW_INSTANCES = 6` yazın (≈ 1,2 saat); defteri iki kez çalıştırmak koşuyu tamamlar. Sonunda `v23_marking12_results.zip` tarayıcıya iner (Drive'da da durur); bu dosyayı gönderin.

In [ ]:
# Colab çalışma ortamı: GPU (L4 veya A100) + Yüksek RAM (12 vCPU). Deney bitene kadar aynı ortam türünü kullanın.
EXPERIMENT_NAME = "v23_marking12"
MAX_NEW_INSTANCES = None      # örn. 6: deneyi kısa oturumlara böler; biten örnekler Drive'dan okunur ve atlanır
SAVE_TO_DRIVE = True
ALLOW_HARDWARE_CHANGE = False  # True: sonraki oturum başka GPU türünde sürebilir (rapora not düşülür)

W = 6                         # paralel LKH süreci (tohum) sayısı
SEEDS = list(range(101, 107))
N_NODES = 10000
PER_FAMILY = 6                # 6 uniform + 6 kümeli = 12 yeni örnek
GENERATOR_SEEDS = {"uniform": 7111, "clustered": 7212}   # önceki hiçbir koşuda kullanılmadı
MARK_COUNT = 5                # sayım işaretlemesi: altı ilk deneme turunun en az bu kadarında bulunan kenarlar alfa = 0 alır
                              # (model işaretlemesinin eşiği, tohumun ilk turunda aynı sayıda kenarı koruyacak biçimde seçilir)
BUDGET_S = 120.0
K_GRAPH = 16
K_LIST = 5
K_LEARNED = 3
PAD_SECONDS = 3.0
REQUIRE_12_VCPUS = True
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
ARM_NAMES = ["pop_full", "pop_native", "pop_model", "beta_tree", "beta_model", "beta_count"]
CODE_SHA256_EXPECTED = "58903205f379cf89e7f102333bac9bfc9abffd6ff816d5e4df3582dce75de495"   # alphaenc_v21.py dosyasının özeti (run v21 ve v22 ile aynı modül)
MARKING_SHA256_EXPECTED = "c4312c50a82f1acfcff4059b7a8bb1c798d53db95cce6c06b86471d05a7591dc"   # marking_v23.py dosyasının özeti (bu koşunun işaretleme modülü)
NOTEBOOK_BUILD = "4fbd5d8ba6e2c446fb3b76161e75202f533d341833b4a3f1d627531513001a43"      # bu defterin bütün kod hücrelerinin özeti

import os

TEST_MODE = os.environ.get("BETATOUR_V23_TEST") == "1"    # yalnızca yerel öz sınama; Colab'da kapalıdır
if TEST_MODE:
    EXPERIMENT_NAME = "v23_selftest"
    SAVE_TO_DRIVE = False
    REQUIRE_12_VCPUS = False
    W = int(os.environ.get("BETATOUR_V23_W", "2"))
    SEEDS = SEEDS[:W]
    N_NODES = int(os.environ.get("BETATOUR_V23_N", "1000"))
    PER_FAMILY = int(os.environ.get("BETATOUR_V23_PER_FAMILY", "1"))
    BUDGET_S = float(os.environ.get("BETATOUR_V23_BUDGET", "6"))
    MARK_COUNT = min(MARK_COUNT, W)
assert W == len(SEEDS) and K_LEARNED < K_LIST <= K_GRAPH

In [ ]:
%%writefile alphaenc_v21.py
"""Run v21: alpha-encoding control. Helper functions embedded in notebook 23.

A candidate file of LKH-3 holds, per node, the father of the node in the minimum 1-tree (Dad) and the candidates
with their alpha-values. LKH-3 uses both fields after it has read the file:

  * Every trial after the first starts from a tour built by a random walk (ChooseInitialTour.c). The walk keeps an
    edge of the best or next best tour only if that edge is a candidate whose alpha-value is zero (case C). With
    CANDIDATE_SET_TYPE = POPMUSIC and SUBGRADIENT = NO, LKH gives alpha = 0 to the edges of its minimum 1-tree, about
    two candidates per node.
  * Until the first tour of a run is recorded, the macro Near(a, b) is true for the edges of the minimum 1-tree
    (a is the father of b or b the father of a); the first trial does not break these edges first.

The candidate files of runs v1-v8 gave alpha = 0 to the first candidate of each node only (alpha = 0, 100, 200, ...)
and wrote no father, so their restarted arms kept fewer edges of the best tour at every trial.

This module holds
  * the two encodings compared in run v21 ("rank": 0, 100, 200, ..., no father; "tree": alpha = 0 exactly on the
    candidates that LKH's own lists of the same seed mark with alpha = 0, and the fathers of those lists),
  * LKH task functions that mirror tsplab.task_first_trial, task_anytime and task_guided, return the alpha-values
    and fathers of LKH's own lists and count the completed trials (TRACE_LEVEL = 2),
  * the share of a tour's edges that LKH's restart keeps under a given candidate file,
  * a self-check that compares LKH's search under its own candidate file and under rewritten files, and
  * the paired statistics of the analysis.
"""

from __future__ import annotations

import os
import re
import shutil
import tempfile
from math import comb

import numpy as np

import tsplab as T

VERSION = "v21-alphaenc-3"

# One line per completed trial at TRACE_LEVEL = 2: "* 12: Cost = 123, Time = 1.23 sec." (new best tour of the run)
# or "  13: Cost = 124, Time = 1.31 sec." (no improvement). LKH reports its own CPU time since the search began.
TRIAL_RX = re.compile(r"^(\*?)[ \t]*(\d+):[ \t]*Cost[ \t]*=[ \t]*(-?\d+)[^\n]*?Time[ \t]*=[ \t]*([\d.]+)[ \t]*sec", re.M)


# ----------------------------------------------------------------------------- encodings
def edge_keys(a, b, n):
    """Undirected edge keys min * n + max for 0-based node arrays a and b."""
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)
    return np.minimum(a, b) * int(n) + np.maximum(a, b)


def tree_edge_keys(cand_native, alpha_native):
    """Keys of the edges that LKH's own lists mark with alpha = 0 (the edges of its minimum 1-tree).

    cand_native: (n, K) 0-based neighbour ids, -1 padded; alpha_native: (n, K) alpha-values written by LKH."""
    cand_native = np.asarray(cand_native, dtype=np.int64)
    alpha_native = np.asarray(alpha_native, dtype=np.int64)
    assert cand_native.shape == alpha_native.shape and cand_native.ndim == 2
    n = cand_native.shape[0]
    rows = np.broadcast_to(np.arange(n, dtype=np.int64)[:, None], cand_native.shape)
    mask = (cand_native >= 0) & (alpha_native == 0)
    return np.unique(edge_keys(rows[mask], cand_native[mask], n))


def rank_alpha(cand):
    """The encoding of runs v1-v8: alpha = 0, 100, 200, ... in list order (tsplab.write_candidates default)."""
    cand = np.asarray(cand)
    return np.broadcast_to(100 * np.arange(cand.shape[1], dtype=np.int64)[None, :], cand.shape).copy()


def tree_alpha(cand, tree_keys):
    """Tree encoding: alpha = 0 for the entries of `cand` whose edge is in `tree_keys`, and 100, 200, ... by list
    position for the other entries. LKH sorts each list by alpha and then by length when it reads the file, so the
    alpha = 0 entries move to the front and the other entries keep their order. Returns (alpha, is_tree)."""
    cand = np.asarray(cand, dtype=np.int64)
    n, k = cand.shape
    rows = np.broadcast_to(np.arange(n, dtype=np.int64)[:, None], cand.shape)
    valid = cand >= 0
    keys = edge_keys(rows, np.where(valid, cand, 0), n)
    is_tree = valid & np.isin(keys, np.asarray(tree_keys, dtype=np.int64))
    alpha = np.where(is_tree, 0, 100 * (np.arange(k, dtype=np.int64)[None, :] + 1))
    return alpha.astype(np.int64), is_tree


def kept_share(cand, alpha, tour):
    """Share of the edges of `tour` that LKH's restart can keep under a candidate file (cand, alpha): the edge (i, j)
    is kept when the walk reaches i and j is a candidate of i with alpha = 0. Mean over both walking directions."""
    cand = np.asarray(cand, dtype=np.int64)
    alpha = np.asarray(alpha, dtype=np.int64)
    tour = np.asarray(tour, dtype=np.int64)
    n = len(tour)
    assert cand.shape[0] == n and cand.shape == alpha.shape
    succ = np.empty(n, dtype=np.int64)
    pred = np.empty(n, dtype=np.int64)
    succ[tour] = np.roll(tour, -1)
    pred[tour] = np.roll(tour, 1)
    zero = (cand >= 0) & (alpha == 0)
    forward = ((cand == succ[:, None]) & zero).any(1).mean()
    backward = ((cand == pred[:, None]) & zero).any(1).mean()
    return float((forward + backward) / 2.0)


# ----------------------------------------------------------------------------- LKH tasks
def trial_times(stdout):
    """End times (LKH's clock, seconds since its search began) of the trials printed at TRACE_LEVEL = 2."""
    rows = [(int(k), float(t)) for _, k, _, t in TRIAL_RX.findall(stdout or "")]
    times = []
    for index, (k, t) in enumerate(rows, 1):
        if k != index:          # a truncated or foreign line: keep the consistent prefix only
            break
        times.append(t)
    return times


def trial_costs(stdout):
    """(trial number, cost) of every trial printed at TRACE_LEVEL = 2, in order."""
    return [(int(k), int(c)) for _, k, c, _ in TRIAL_RX.findall(stdout or "")]


def trials_within(times, offset, budget, fractions=(0.25, 0.5, 0.75, 1.0)):
    """Number of completed trials whose absolute end time (offset + LKH time) is within each fraction of the budget."""
    absolute = np.asarray(times, dtype=np.float64) + float(offset)
    return [int((absolute <= f * budget + 1e-9).sum()) for f in fractions]


def task_first_trial(lkh_bin, workroot, seed, xy, problem_file=None, ew="EUC_2D", k=T.CAND_K, timeout=None):
    """tsplab.task_first_trial (ascent-free POPMUSIC lists and one trial) at TRACE_LEVEL = 2 that also returns the
    alpha-values and the fathers (0-based, -1 for the root) of the run's own candidate lists, as LKH wrote them to
    its CANDIDATE_FILE. tsplab reads each list in ascending alpha order, which is the order of LKH's file."""
    p = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_TRIALS": 1, "MAX_CANDIDATES": int(k),
         "SEED": int(seed), "TRACE_LEVEL": 2}
    res = T.lkh_solve(lkh_bin, xy=None if problem_file else xy, problem_file=problem_file, ew=ew, params=p,
                      workroot=workroot, want_cands=True, timeout=timeout)
    n = len(xy)
    if not T.is_valid_tour(res["tour"], n):
        raise RuntimeError(f"first trial gave no valid tour (rc={res['returncode']}): {res['stdout'][-500:]}")
    if "cand" not in res:
        raise RuntimeError("LKH did not write the candidate file: " + res["stdout"][-500:])
    cand, alpha, dad = res["cand"]["cand"], res["cand"]["alpha"], np.asarray(res["cand"]["dad"], dtype=np.int64)
    if cand.shape != (n, k) or alpha.shape != (n, k) or dad.shape != (n,):
        raise RuntimeError(f"unexpected candidate list shape {cand.shape}")       # no silent truncation to k
    return {"tour": res["tour"].astype(np.int32), "cost": float(T.tour_length(xy, res["tour"], ew)),
            "cand": cand.astype(np.int32), "alpha": alpha.astype(np.int64), "dad": dad, "events": res["events"],
            "pre_time": res["pre_time"], "wall": res["wall"], "t_start": res["t_start"]}


def task_anytime(lkh_bin, workroot, budget, seed, xy=None, problem_file=None, ew="EUC_2D", params=None,
                 timeout=None):
    """tsplab.task_anytime with TRACE_LEVEL = 2; also returns the end time of every completed trial. At this trace
    level POPMUSIC prints its own "Cost.min" and "Cand.min" lines first, so tsplab's cost_min and cand_stats would
    describe the POPMUSIC sample tours; they are not returned."""
    p = {"TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000, "SEED": int(seed), "TRACE_LEVEL": 2}
    p.update(params or {})
    res = T.lkh_solve(lkh_bin, xy=xy, problem_file=problem_file, ew=ew, params=p, workroot=workroot,
                      timeout=budget * 3 + 600 if timeout is None else timeout, keep_stdout=True)
    out = {k: res.get(k) for k in ("events", "pre_time", "wall", "returncode", "timed_out", "t_start")}
    out["trial_times"] = trial_times(res.get("stdout"))
    return out


def task_guided(lkh_bin, workroot, budget, seed, xy, cand, alpha=None, dad=None, init_tour=None, problem_file=None,
                ew="EUC_2D", extra_params=None, timeout=None):
    """tsplab.task_guided without fixed edges (candidate file with the given alpha-values and fathers, zero
    penalties, initial tour, no ascent) with TRACE_LEVEL = 2; also returns the end time of every completed trial.
    alpha = None writes 0, 100, 200, ... and dad = None writes no father, as in runs v1-v8."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="g_")
    try:
        n = len(xy)
        pf = problem_file
        if pf is None:
            pf = os.path.join(wd, "p.tsp")
            T.write_tsp(pf, xy, ew=ew)
        cf, pif = os.path.join(wd, "g.cand"), os.path.join(wd, "g.pi")
        T.write_candidates(cf, cand, alpha, dad)
        T.write_pi(pif, np.zeros(n))
        params = {"SUBGRADIENT": "NO", "TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000,
                  "SEED": int(seed), "MAX_CANDIDATES": int(np.asarray(cand).shape[1]), "TRACE_LEVEL": 2}
        if init_tour is not None:
            tp = os.path.join(wd, "init.tour")
            T.write_tour(tp, init_tour)
            params["INITIAL_TOUR_FILE"] = tp
        params.update(extra_params or {})
        params["CANDIDATE_FILE"], params["PI_FILE"] = cf, pif
        res = T.run_lkh(lkh_bin, pf, params, os.path.join(wd, "run"),
                        timeout=budget * 3 + 600 if timeout is None else timeout, keep_stdout=True)
        ok = T.is_valid_tour(res["tour"], n)
        out = {k: res.get(k) for k in ("events", "pre_time", "wall", "returncode", "timed_out", "t_start")}
        out["cost_check"] = float(T.tour_length(xy, res["tour"], ew)) if ok else None
        out["trial_times"] = trial_times(res.get("stdout"))
        if not ok:
            out["stdout_tail"] = (res.get("stdout") or "")[-800:]
        return out
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def encoding_selfcheck(lkh_bin, workroot, xy, seed, trials=25, k=T.CAND_K, ew="EUC_2D", timeout=300):
    """Compares, on the machine of the run, LKH's search under the candidate file it wrote itself with its search
    under four rewritten files: LKH's own alpha-values or 0, 100, 200, ..., each with and without the fathers.
    All continuations start from the tour of the first trial, have a fixed number of trials and no time limit, so
    their traces are deterministic. Expected from the source code:
      * own alpha-values with fathers: the same cost at every trial as under LKH's own file;
      * 0, 100, 200, ... with fathers: a different search from the second trial on (restart walk, case C);
      * without fathers: a difference, if any, can start in the first trial (Near() before a tour is recorded).
    Also checks that the tree rule reproduces the entries that LKH marks with alpha = 0."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="enc_")
    try:
        n = len(xy)
        pf, own = os.path.join(wd, "p.tsp"), os.path.join(wd, "own.cand")
        T.write_tsp(pf, xy, ew=ew)
        first = T.run_lkh(lkh_bin, pf, {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_TRIALS": 1,
                                        "MAX_CANDIDATES": int(k), "SEED": int(seed), "TRACE_LEVEL": 2,
                                        "CANDIDATE_FILE": own}, os.path.join(wd, "first"), timeout=timeout)
        if first["returncode"] != 0 or not T.is_valid_tour(first["tour"], n) or not os.path.exists(own):
            raise RuntimeError("self-check: the first trial failed: " + (first.get("stdout") or "")[-500:])
        lists = T.read_candidates(own, k_pad=k)
        cand, alpha, dad = lists["cand"], lists["alpha"], lists["dad"]
        if cand.shape != (n, k):
            raise RuntimeError(f"self-check: unexpected candidate list shape {cand.shape}")
        pif, tp = os.path.join(wd, "zero.pi"), os.path.join(wd, "first.tour")
        T.write_pi(pif, np.zeros(n))
        T.write_tour(tp, first["tour"])
        variants = {"native_dad": (alpha, dad), "native_nodad": (alpha, None),
                    "rank_dad": (rank_alpha(cand), dad), "rank_nodad": (rank_alpha(cand), None)}
        files = {"lkh_file": own}
        for name, (a, d) in variants.items():
            files[name] = os.path.join(wd, name + ".cand")
            T.write_candidates(files[name], cand, a, d)
        traces = {}
        for name, cf in files.items():
            res = T.run_lkh(lkh_bin, pf, {"SUBGRADIENT": "NO", "MAX_TRIALS": int(trials), "MAX_CANDIDATES": int(k),
                                          "SEED": int(seed), "TRACE_LEVEL": 2, "INITIAL_TOUR_FILE": tp,
                                          "CANDIDATE_FILE": cf, "PI_FILE": pif}, os.path.join(wd, name),
                            timeout=timeout, keep_stdout=True)
            traces[name] = trial_costs(res.get("stdout"))
            if res["returncode"] != 0 or [k_ for k_, _ in traces[name]] != list(range(1, int(trials) + 1)):
                raise RuntimeError(f"self-check: continuation '{name}' failed: " + (res.get("stdout") or "")[-500:])

        def first_difference(name):
            trials_ = [k_ for (k_, a), (_, b) in zip(traces[name], traces["lkh_file"]) if a != b]
            return int(trials_[0]) if trials_ else None

        valid = cand >= 0
        _, is_tree = tree_alpha(cand, tree_edge_keys(cand, alpha))
        return {"nodes": int(n), "trials": int(trials), "seed": int(seed),
                "native_file_equals_lkh_file": traces["native_dad"] == traces["lkh_file"],
                "first_trial_with_a_different_cost": {name: first_difference(name) for name in variants},
                "tree_rule_matches_lkh": bool(np.array_equal(is_tree, valid & (alpha == 0))),
                "candidates_per_node": float(valid.sum() / n),
                "zero_alpha_per_node": float((valid & (alpha == 0)).sum() / n),
                "nodes_with_a_father": int((dad >= 0).sum()),
                "kept_share_native": kept_share(cand, alpha, first["tour"]),
                "kept_share_rank": kept_share(cand, rank_alpha(cand), first["tour"]),
                "best_cost": {name: int(min(c for _, c in trace)) for name, trace in traces.items()}}
    finally:
        shutil.rmtree(wd, ignore_errors=True)


# ----------------------------------------------------------------------------- statistics
def sign_test_p(wins, losses):
    """Exact two-sided sign test."""
    n, k = int(wins) + int(losses), min(int(wins), int(losses))
    if n == 0:
        return 1.0
    return float(min(1.0, 2.0 * sum(comb(n, j) for j in range(k + 1)) / 2.0 ** n))


def paired_stats(values, bootstrap_seed, replicates=5000):
    """Paired percentage differences (first minus second arm): wins are negative values (the first arm is shorter)."""
    x = np.asarray(values, dtype=np.float64)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    wins, losses = int((x < -1e-12).sum()), int((x > 1e-12).sum())
    rng = np.random.default_rng(bootstrap_seed)
    means = x[rng.integers(0, len(x), size=(replicates, len(x)))].mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return dict(instances=int(len(x)), win=wins, tie=int(len(x) - wins - losses), loss=losses,
                mean_delta_pct=float(x.mean()), median_delta_pct=float(np.median(x)),
                ci95_low_pct=float(low), ci95_high_pct=float(high), sign_p=sign_test_p(wins, losses))


def holm(p_values):
    """Holm-adjusted p-values in the input order."""
    p = np.asarray(p_values, dtype=np.float64)
    order = np.argsort(p, kind="stable")
    adjusted = np.empty(len(p))
    running = 0.0
    for rank, j in enumerate(order):
        running = max(running, (len(p) - rank) * p[j])
        adjusted[j] = min(1.0, running)
    return [float(v) for v in adjusted]


In [ ]:
%%writefile marking_v23.py
"""Run v23: restart markings. Helper functions embedded in notebook 25.

LKH-3 starts every trial after the first from a tour built by a random walk (ChooseInitialTour.c). The walk follows
an edge of the best or next best tour by rule only if that edge is a candidate whose alpha-value is zero; elsewhere
it moves to a random unvisited candidate. With restricted search (the default), a move of the trial can start only at
an edge that is not in the best tour. The entries with alpha = 0 in a candidate file therefore decide which edges of
the best tour every trial keeps and where its search starts.

LKH-3 itself gives alpha = 0 to the edges of its minimum 1-tree. This module writes other markings on given candidate
lists:

  * count marking: alpha = 0 on the entries whose edge is in at least a given number of the ensemble tours, and
  * model marking: alpha = 0 on the entries whose edge has a predicted probability of at least a threshold; the
    threshold is set so that the marking trusts as many edges of a given tour as the count marking does.

The lists themselves and the fathers (Dad) of the file are not changed by a marking.
"""

from __future__ import annotations

import numpy as np

VERSION = "v23-marking-1"


def edge_keys(a, b, n):
    """Undirected edge keys min * n + max for 0-based node arrays a and b (as alphaenc.edge_keys)."""
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)
    return np.minimum(a, b) * int(n) + np.maximum(a, b)


def entry_keys(cand):
    """Edge key of every entry of the lists `cand` ((n, K) 0-based neighbour ids, -1 padded); -1 for empty entries."""
    cand = np.asarray(cand, dtype=np.int64)
    assert cand.ndim == 2
    n = cand.shape[0]
    rows = np.broadcast_to(np.arange(n, dtype=np.int64)[:, None], cand.shape)
    valid = cand >= 0
    return np.where(valid, edge_keys(rows, np.where(valid, cand, 0), n), -1)


def tour_edge_keys(tour):
    """Sorted keys of the n edges of a tour given as a permutation of 0..n-1."""
    tour = np.asarray(tour, dtype=np.int64)
    return np.unique(edge_keys(tour, np.roll(tour, -1), len(tour)))


def key_count(keys, tours):
    """For every edge key, the number of the given tours that contain the edge (0 for the key -1)."""
    keys = np.asarray(keys, dtype=np.int64)
    count = np.zeros(keys.shape, dtype=np.int64)
    for tour in tours:
        count += np.isin(keys, tour_edge_keys(tour))
    return count


def ensemble_count(cand, tours):
    """For every entry of `cand`, the number of the given tours that contain its edge (0 for empty entries)."""
    return key_count(entry_keys(cand), tours)


def probability_table(prob, nb):
    """Undirected edge probabilities from the directed output of the model.

    prob[i, j] is the predicted probability of the edge from node i to its graph neighbour nb[i, j]. The probability
    of an undirected edge is the maximum over the directions in which the graph holds it. Returns the sorted edge
    keys and their probabilities."""
    nb = np.asarray(nb, dtype=np.int64)
    prob = np.asarray(prob, dtype=np.float64)
    assert nb.ndim == 2 and prob.shape == nb.shape and np.all(nb >= 0)
    n, k = nb.shape
    keys = edge_keys(np.repeat(np.arange(n, dtype=np.int64), k), nb.reshape(-1), n)
    order = np.argsort(keys, kind="stable")
    keys = keys[order]
    unique, start = np.unique(keys, return_index=True)
    return unique, np.maximum.reduceat(prob.reshape(-1)[order], start)


def key_probability(keys, table):
    """Probability of every edge key from probability_table(); 0 for the key -1 and for edges that the graph of the
    model does not hold."""
    unique, probability = table
    keys = np.asarray(keys, dtype=np.int64)
    pos = np.clip(np.searchsorted(unique, keys), 0, len(unique) - 1)
    return np.where((keys >= 0) & (unique[pos] == keys), probability[pos], 0.0)


def entry_probability(cand, table):
    """Probability of every entry of `cand` (0 for empty entries and for edges outside the graph of the model)."""
    return key_probability(entry_keys(cand), table)


def matched_threshold(tour_probability, keep):
    """Probability threshold that trusts `keep` edges of a tour: the probability of the tour's keep-th most probable
    edge (infinity for keep = 0). Edges with exactly that probability are trusted together."""
    p = np.sort(np.asarray(tour_probability, dtype=np.float64))[::-1]
    keep = int(keep)
    assert p.ndim == 1 and 0 <= keep <= len(p)
    return float("inf") if keep == 0 else float(p[keep - 1])


def trusted_by_probability(probability, threshold):
    """Entries trusted by the model marking: probability at least the threshold; an edge outside the graph of the
    model (probability 0) is never trusted."""
    probability = np.asarray(probability, dtype=np.float64)
    return (probability >= float(threshold)) & (probability > 0.0)


def mark_alpha(cand, trusted, rest=None):
    """Alpha-values of a marking: 0 on the trusted entries of `cand`; the other entries keep their order behind them.

    rest: alpha-values that order the other entries (for example LKH's own; values below 1 are raised to 1). Without
    it the other entries get 100, 200, ... by list position, as in the tree encoding of run v21. LKH sorts each list
    by alpha-value and then by length when it reads the file."""
    cand = np.asarray(cand, dtype=np.int64)
    trusted = np.asarray(trusted, dtype=bool)
    assert cand.ndim == 2 and trusted.shape == cand.shape
    if rest is None:
        rest = np.broadcast_to(100 * (np.arange(cand.shape[1], dtype=np.int64)[None, :] + 1), cand.shape)
    rest = np.asarray(rest, dtype=np.int64)
    assert rest.shape == cand.shape
    return np.where(trusted & (cand >= 0), 0, np.maximum(rest, 1)).astype(np.int64)


In [ ]:
import hashlib
import importlib.util
import json
import math
import platform
import shutil
import subprocess
import sys
import tarfile
import time
import traceback
import urllib.request
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

for package, module in [("scipy", "scipy"), ("pandas", "pandas")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import scipy
import torch

torch.set_num_threads(1)
if TEST_MODE:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    assert torch.cuda.is_available(), "Colab'da GPU çalışma ortamı seçin: L4 veya A100."
    DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= W, f"{W} LKH süreci için en az {W} vCPU gerekli; görünen: {CPU_COUNT}"
if REQUIRE_12_VCPUS:
    assert CPU_COUNT >= 2 * W, f"Makaledeki düzen için 12 vCPU gerekli (Yüksek RAM seçin); görünen: {CPU_COUNT}"


def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"


CPU_MODEL = cpu_model_name()
WORK_ROOT = Path(os.environ.get("BETATOUR_V23_WORK", "/content/betatour_v23_work"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Depo: run v5-v22 ile aynı sabit commit (model kodu ve eğitilmiş model).
if TEST_MODE and os.environ.get("BETATOUR_V23_REPO"):
    REPO = Path(os.environ["BETATOUR_V23_REPO"])
else:
    REPO = WORK_ROOT / "repo"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--quiet",
                        "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
    assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M

# Bu deneyin iki yardımcı modülü (önceki iki hücrenin yazdığı dosyalar). Dosya sonundaki satır sonu sayısı arayüze
# göre değişebildiği için tek satır sonuna indirilir; içerik depodaki modülle bayt bayt aynı olmalıdır.
def embedded_module(file_name, expected_sha256):
    path = Path(file_name).resolve()
    assert path.exists(), f"Önce {file_name} dosyasını yazan hücreyi çalıştırın."
    data = path.read_bytes().rstrip(b"\n") + b"\n"
    digest = hashlib.sha256(data).hexdigest()
    assert digest == expected_sha256, f"{file_name} depodaki modülle aynı değil; o hücreyi değiştirmeden çalıştırın."
    if path.read_bytes() != data:
        path.write_bytes(data)
    return path, digest


CODE_PATH, CODE_SHA256 = embedded_module("alphaenc_v21.py", CODE_SHA256_EXPECTED)
MARKING_PATH, MARKING_SHA256 = embedded_module("marking_v23.py", MARKING_SHA256_EXPECTED)
sys.path.insert(0, str(CODE_PATH.parent))
import alphaenc_v21 as A
import marking_v23 as K

checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
assert hashlib.sha256(checkpoint_path.read_bytes()).hexdigest() == CHECKPOINT_SHA256, \
    "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)   # sabitlenmiş, güvenilen depo
assert MODEL.cfg.get("use_ensemble") is True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "stages", "candidates", "errors", "summary", "code"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
shutil.copy2(CODE_PATH, OUTPUT_ROOT / "code" / "alphaenc_v21.py")
shutil.copy2(MARKING_PATH, OUTPUT_ROOT / "code" / "marking_v23.py")


def get_lkh_3014():
    """Resmi kaynak; ulaşılamazsa SHA256 denetimli sabit kopya (run v10-v12 ile aynı)."""
    if TEST_MODE and os.environ.get("BETATOUR_V23_LKH"):
        return os.environ["BETATOUR_V23_LKH"], "local test binary"
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    marker = root / "source.txt"
    if binary.exists() and marker.exists():
        return str(binary), marker.read_text().strip()
    root.mkdir(parents=True, exist_ok=True)
    options = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned copy", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
         "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    f.write(block)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH kopyası SHA256 uyuşmazlığı: {got}"
            shutil.rmtree(root / "LKH-3.0.14", ignore_errors=True)
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            assert (root / "LKH-3.0.14" / "Makefile").exists(), "LKH kaynak klasörü eksik"
            build = subprocess.run(["make", "-j", str(min(4, CPU_COUNT))], cwd=root / "LKH-3.0.14",
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=900)
            if build.returncode != 0:
                raise RuntimeError("LKH derlemesi başarısız:\n" + build.stdout[-3000:])
            assert binary.exists(), "LKH ikilisi oluşmadı"
            marker.write_text(f"{label}: {url}")
            return str(binary), f"{label}: {url}"
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi veya derlenemedi: {last_error}")


LKH, LKH_SOURCE = get_lkh_3014()
# Deneyin konusu olan iki kural derlenen kaynakta var mı? (ChooseInitialTour.c, C durumu; LKH.h, Near makrosu)
walk_source = Path(LKH).parent / "SRC" / "ChooseInitialTour.c"
near_source = Path(LKH).parent / "SRC" / "INCLUDE" / "LKH.h"
WALK_SHA256 = None
if walk_source.exists() and near_source.exists():
    assert "NN->Alpha == 0 && (InBestTour(N, NextN) ||" in walk_source.read_text(errors="replace"), \
        "ChooseInitialTour.c beklenen kuralı içermiyor"
    assert "(a)->BestSuc ? InBestTour(a, b) : (a)->Dad == (b) || (b)->Dad == (a)" in near_source.read_text(errors="replace"), \
        "LKH.h beklenen Near makrosunu içermiyor"
    WALK_SHA256 = hashlib.sha256(walk_source.read_bytes()).hexdigest()

HARDWARE = {
    "gpu": GPU_NAME,
    "vcpus_visible": CPU_COUNT,
    "cpu_model": CPU_MODEL,
    "ram_gb": T.env_info().get("ram_gb"),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
}
MANIFEST = {
    "schema": 1,
    "experiment_name": EXPERIMENT_NAME,
    "test_mode": TEST_MODE,
    "design": {"nodes": N_NODES, "per_family": PER_FAMILY, "generator_seeds": GENERATOR_SEEDS,
               "budget_s": BUDGET_S, "pad_seconds": PAD_SECONDS, "mark_count": MARK_COUNT},
    "seeds": SEEDS,
    "workers": W,
    "graph_k": K_GRAPH,
    "candidate_k": K_LIST,
    "learned_k": K_LEARNED,
    "arms": ARM_NAMES,
    "repo_commit": REPO_COMMIT,
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "code_version": A.VERSION,
    "code_sha256": CODE_SHA256,
    "marking_version": K.VERSION,
    "marking_sha256": MARKING_SHA256,
    "notebook_build": NOTEBOOK_BUILD,
    "lkh_source": LKH_SOURCE,
    "choose_initial_tour_sha256": WALK_SHA256,
    "hardware": HARDWARE,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "design", "seeds", "workers", "graph_k", "candidate_k", "learned_k", "arms",
                "repo_commit", "checkpoint_sha256", "code_version", "code_sha256", "marking_version",
                "marking_sha256", "notebook_build"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı bir EXPERIMENT_NAME kullanın."
    assert saved["hardware"]["vcpus_visible"] == HARDWARE["vcpus_visible"], "vCPU sayısı değişti; aynı ortam türünü seçin."
    assert ALLOW_HARDWARE_CHANGE or saved["hardware"]["gpu"] == HARDWARE["gpu"], \
        (f"GPU türü değişti ({saved['hardware']['gpu']} -> {HARDWARE['gpu']}). Aynı türü seçin; bulunamıyorsa ilk "
         "hücrede ALLOW_HARDWARE_CHANGE = True yapın (her örneğin bütün kolları yine tek makinede koşar).")
    if saved["hardware"] != HARDWARE:
        print("UYARI: donanım veya kütüphane sürümleri ilk oturumdan farklı; her örneğin donanımı ayrıca kaydedilir.")
else:
    manifest_path.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

# Aynı Colab sanal makinesinde defter yeniden çalıştırılırsa bu işaret korunur. Yeni makinede yarım kalan
# örnek baştan çalıştırılır; böylece bir örneğin bütün kolları tek makinede koşar.
session_path = WORK_ROOT / "runtime_session_id.txt"
if session_path.exists():
    SESSION_ID = session_path.read_text().strip()
else:
    SESSION_ID = uuid.uuid4().hex
    session_path.write_text(SESSION_ID)

print("Hazır:", GPU_NAME, "|", CPU_COUNT, "vCPU |", sum(p.numel() for p in MODEL.parameters()), "model parametresi |",
      "LKH:", LKH_SOURCE, "| kod", A.VERSION, CODE_SHA256[:12], K.VERSION, MARKING_SHA256[:12],
      "| sonuçlar:", OUTPUT_ROOT)

In [ ]:
def make_instances():
    """12 yeni sentetik EUC_2D örneği (10.000 düğüm): 6 uniform ve 6 kümeli; aileler dönüşümlü sıralanır."""
    result = []
    for family, group in (("uniform", "uni10k"), ("clustered", "clu10k")):
        rng = np.random.default_rng(GENERATOR_SEEDS[family])
        for i in range(PER_FAMILY):
            xy = T.to_int_coords(T.gen_instance(N_NODES, family, rng))
            result.append(dict(group=group, family=family, index=i, name=f"mark_{group}_{i:03d}", xy=xy,
                               ew="EUC_2D", problem_file=None, budget=BUDGET_S))
    result.sort(key=lambda x: (x["index"], x["group"]))
    assert len({x["name"] for x in result}) == len(result) == 2 * PER_FAMILY
    return result


INSTANCES = make_instances()
# Run v5'te 10.000 düğümlü örneklerde ölçülen süreler: ortak ilk aşama 12-14 s, yeniden başlatılan kol 108-111 s.
phase_s = 15.0 if N_NODES >= 10000 else 2.0
estimate_h = len(INSTANCES) * (phase_s + BUDGET_S + PAD_SECONDS
                               + (len(ARM_NAMES) - 1) * (BUDGET_S - phase_s + PAD_SECONDS + 2.0)) / 3600.0
print(f"{len(INSTANCES)} örnek hazır; {len(ARM_NAMES)} kol için tahmini toplam süre yaklaşık {estimate_h:.1f} saat "
      f"(örnek başına yaklaşık {estimate_h * 60 / len(INSTANCES):.1f} dk).")

In [ ]:
POP_PARAMS = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_LIST}
RESTART_ARMS = [arm for arm in ARM_NAMES if arm != "pop_full"]
FRACTIONS = [0.25, 0.5, 0.75, 1.0]
# Yeniden başlatılan her kolun aday dosyası: (aday dizisi, alfa dizisi, baba alanı yazılsın mı).
# Diziler candidates/<örnek>.npz içindedir; baba (Dad) alanı ilk aşamanın pop_dad dizisinden gelir.
# İki liste (POPMUSIC, BetaTour karma) x iki işaretleme (LKH'nin 1-ağacı, model) ve BetaTour listesinde sayım.
ARM_FILES = {
    "pop_native": ("cand_pop", "alpha_pop_native", True),    # LKH'nin kendi dosyasının eşdeğeri (ağaç işaretlemesi)
    "pop_model": ("cand_pop", "alpha_pop_model", True),      # aynı liste, model işaretlemesi
    "beta_tree": ("cand_beta", "alpha_beta_tree", True),     # BetaTour karma listesi, ağaç işaretlemesi (run v21)
    "beta_model": ("cand_beta", "alpha_beta_model", True),   # aynı liste, model işaretlemesi
    "beta_count": ("cand_beta", "alpha_beta_count", True),   # aynı liste, sayım işaretlemesi
}
assert set(ARM_FILES) == set(RESTART_ARMS)


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)


def write_npz_atomic(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def parallel_tasks(func, kwargs):
    with ThreadPoolExecutor(max_workers=W) as pool:
        futures = [pool.submit(func, **kw) for kw in kwargs]
        return [future.result() for future in futures]


def instance_common(inst):
    return dict(lkh_bin=LKH, workroot=str(WORK_ROOT / "lkh_tmp"), xy=inst["xy"],
                problem_file=inst["problem_file"], ew=inst["ew"])


def event_list(res, offset):
    return [[int(cost), float(t)] for cost, t in T.abs_events(res, offset=offset)]


def best_within(events, limit):
    costs = [int(cost) for cost, t in events if t <= limit + 1e-9]
    return min(costs) if costs else None


def run_record(label, seed, events, trial_end, budget, **extra):
    """Bir sürecin kaydı. Bütün zamanlar örneğin başlangıcından itibaren saniyedir (ortak saat)."""
    best = best_within(events, budget)
    assert best is not None, f"{label} / tohum {seed}: bütçe içinde tur yok"
    trial_end = [round(float(t), 3) for t in trial_end]
    return dict(seed=int(seed), events=events, best_at_budget=int(best),
                best_at_fraction=[best_within(events, f * budget) for f in FRACTIONS],
                trials_at_fraction=[int(sum(t <= f * budget + 1e-9 for t in trial_end)) for f in FRACTIONS],
                trial_end_s=trial_end, **extra)


def arm_record(runs, **extra):
    at_fraction = []
    for j in range(len(FRACTIONS)):
        known = [r["best_at_fraction"][j] for r in runs if r["best_at_fraction"][j] is not None]
        at_fraction.append(min(known) if known else None)
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                mean_at_budget=float(np.mean([r["best_at_budget"] for r in runs])),
                best_at_fraction=at_fraction,
                mean_trials_at_fraction=[float(np.mean([r["trials_at_fraction"][j] for r in runs]))
                                         for j in range(len(FRACTIONS))], **extra)


def exit_ok(res):
    return res["returncode"] == 0 or (res["returncode"] is None and res["timed_out"])


def make_full_arm(inst):
    """Kesintisiz POPMUSIC-5: altı süreç, bütçe sonunda dış saatle durdurulur (run v5'teki pop_full)."""
    budget = inst["budget"]
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(A.task_anytime, [dict(common, budget=budget, seed=seed, params=POP_PARAMS,
                                               timeout=budget + PAD_SECONDS) for seed in SEEDS])
    wall = time.time() - start
    runs = []
    for seed, res in zip(SEEDS, raw):
        assert exit_ok(res), f"pop_full / tohum {seed}: LKH çıkış kodu {res['returncode']}"
        assert res["pre_time"] is not None, f"pop_full / tohum {seed}: LKH ön işleme süresini yazmadı"
        offset = float(res["t_start"] - start)
        search_start = offset + float(res["pre_time"])
        runs.append(run_record("pop_full", seed, event_list(res, offset),
                               [search_start + t for t in res["trial_times"]], budget,
                               search_start_s=search_start, wall_s=float(res["wall"]),
                               returncode=res["returncode"], timed_out=bool(res["timed_out"])))
    return arm_record(runs, wall_s=float(wall))


def make_phase1(inst):
    """Ortak ilk aşama: her tohum için POPMUSIC-5 listeleri ve tek deneme. Süreç burada durur."""
    n = len(inst["xy"])
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(A.task_first_trial, [dict(common, seed=seed, k=K_LIST,
                                                    timeout=inst["budget"] + PAD_SECONDS) for seed in SEEDS])
    phase_wall = time.time() - start
    tours, cands, alphas, dads, costs, first_events = [], [], [], [], [], []
    for seed, res in zip(SEEDS, raw):
        assert T.is_valid_tour(res["tour"], n), f"ilk tur geçersiz: tohum {seed}"
        assert res["cand"].shape == (n, K_LIST) == res["alpha"].shape, f"aday boyutu yanlış: tohum {seed}"
        assert res["dad"].shape == (n,) and np.all(res["dad"] < n), f"baba alanı beklenmedik: tohum {seed}"
        events = event_list(res, res["t_start"] - start)
        assert len(events) == 1, f"ilk aşamada tek deneme bekleniyordu: tohum {seed}, {len(events)} kayıt"
        assert events[0][0] == int(round(res["cost"])), f"LKH'nin yazdığı maliyet turun uzunluğuna eşit değil: {seed}"
        assert events[0][1] <= inst["budget"], f"ilk deneme bütçeyi aştı: tohum {seed}"
        tours.append(res["tour"])
        cands.append(res["cand"])
        alphas.append(res["alpha"])
        dads.append(res["dad"])
        costs.append(res["cost"])
        first_events.append(events)
    arrays = dict(tours=np.stack(tours).astype(np.int32), pop_candidates=np.stack(cands).astype(np.int32),
                  pop_alpha=np.stack(alphas).astype(np.int64), pop_dad=np.stack(dads).astype(np.int32),
                  costs=np.asarray(costs, dtype=np.float64))
    meta = dict(wall_s=float(phase_wall), first_events=first_events, first_costs=[float(c) for c in costs],
                pre_time_s=[float(res["pre_time"] or 0.0) for res in raw],
                process_wall_s=[float(res["wall"]) for res in raw],
                nodes_with_father=[int((res["dad"] >= 0).sum()) for res in raw])
    return arrays, meta


def valid_candidates(cands, n):
    assert cands.shape == (W, n, K_LIST), cands.shape
    assert np.all((cands >= 0) & (cands < n)), "eksik veya geçersiz aday"
    assert not np.any(cands == np.arange(n)[None, :, None]), "düğüm kendi adayı olamaz"
    for i in range(K_LIST):
        for j in range(i + 1, K_LIST):
            assert not np.any(cands[:, :, i] == cands[:, :, j]), "tekrarlı aday"


def make_candidate_lists(inst, phase):
    """Aday dosyaları. Listeler run v21 ile aynıdır; yenilik, hangi girdilerin alfa = 0 aldığıdır (işaretleme).

    tree  : aynı tohumun kendi POPMUSIC listesinde alfası 0 olan kenarlar (LKH'nin en küçük 1-ağacı) alfa = 0.
    count : altı ilk deneme turunun en az MARK_COUNT tanesinde bulunan kenarlar alfa = 0.
    model : modelin tahmin olasılığı bir eşiğin üstünde olan kenarlar alfa = 0. Eşik tohum başına seçilir: tohumun
            ilk turunda, sayım işaretlemesinin koruduğu kadar kenar korunur (eşit açıklık; fark yalnızca hangi
            kenarların seçildiğindedir).
    Diğer girdiler sıralarını korur. Baba (Dad) alanı her dosyada aynı tohumun kendi dosyasından gelir.
    """
    xy = inst["xy"]
    n = len(xy)
    tours, raw_pop, pop_alpha, costs = phase["tours"], phase["pop_candidates"], phase["pop_alpha"], phase["costs"]
    present = raw_pop >= 0
    assert raw_pop.shape == (W, n, K_LIST) == pop_alpha.shape
    assert np.all(present | (raw_pop == -1)) and np.all(raw_pop < n)
    assert not np.any(raw_pop == np.arange(n)[None, :, None])
    assert np.all(pop_alpha[present] >= 0) and np.all(np.diff(np.where(present, pop_alpha, 2**62), axis=2) >= 0)

    start = time.perf_counter()
    succ = np.stack([T.tour_succ_pred(tour)[0] for tour in tours], axis=1).astype(np.int32)
    successor_s = time.perf_counter() - start

    start = time.perf_counter()
    prediction = M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=costs)
    predict_s = time.perf_counter() - start
    start = time.perf_counter()
    beta = np.stack([T.mix_candidates(prediction["rank"], raw_pop[m], k_first=K_LEARNED, k=K_LIST,
                                      fill=prediction["rank"][:, K_LEARNED:]) for m in range(W)]).astype(np.int32)
    beta_mix_s = time.perf_counter() - start
    valid_candidates(beta, n)

    start = time.perf_counter()
    tree_keys = [A.tree_edge_keys(raw_pop[m], pop_alpha[m]) for m in range(W)]
    beta_tree = np.stack([A.tree_alpha(beta[m], tree_keys[m])[0] for m in range(W)])
    beta_tree_s = time.perf_counter() - start

    # Sayım işaretlemesi: kenarın bulunduğu ilk deneme turu sayısı (tohumun kendi turu dahil).
    start = time.perf_counter()
    beta_count = np.stack([K.mark_alpha(beta[m], K.ensemble_count(beta[m], tours) >= MARK_COUNT) for m in range(W)])
    beta_count_s = time.perf_counter() - start

    # Model işaretlemesi: yönlü tahminlerden yönsüz kenar olasılığı (iki yönün büyüğü); eşik, tohumun ilk turunda
    # sayım işaretlemesinin koruduğu kadar kenarı koruyacak biçimde seçilir.
    start = time.perf_counter()
    table = K.probability_table(prediction["prob"], prediction["NB"])
    own_keys = [K.edge_keys(tours[m], np.roll(tours[m], -1), n) for m in range(W)]
    own_count = [K.key_count(own_keys[m], tours) for m in range(W)]
    own_prob = [K.key_probability(own_keys[m], table) for m in range(W)]
    threshold = [K.matched_threshold(own_prob[m], int((own_count[m] >= MARK_COUNT).sum())) for m in range(W)]
    threshold_s = time.perf_counter() - start
    start = time.perf_counter()
    pop_model = np.stack([K.mark_alpha(raw_pop[m], K.trusted_by_probability(K.entry_probability(raw_pop[m], table),
                                                                            threshold[m]), rest=pop_alpha[m])
                          for m in range(W)])
    pop_model_s = time.perf_counter() - start
    start = time.perf_counter()
    beta_model = np.stack([K.mark_alpha(beta[m], K.trusted_by_probability(K.entry_probability(beta[m], table),
                                                                          threshold[m])) for m in range(W)])
    beta_model_s = time.perf_counter() - start

    lists = dict(cand_pop=raw_pop.astype(np.int32), alpha_pop_native=pop_alpha.astype(np.int64),
                 alpha_pop_model=pop_model, cand_beta=beta, alpha_beta_tree=beta_tree,
                 alpha_beta_model=beta_model, alpha_beta_count=beta_count)
    # Her kol, tek başına koşsaydı ödeyeceği hazırlık süresini öder.
    overhead = dict(pop_native=0.0,
                    pop_model=successor_s + predict_s + threshold_s + pop_model_s,
                    beta_tree=successor_s + predict_s + beta_mix_s + beta_tree_s,
                    beta_model=successor_s + predict_s + beta_mix_s + threshold_s + beta_model_s,
                    beta_count=successor_s + predict_s + beta_mix_s + beta_count_s)

    # Tanılar (süreye dahil değildir). Ağaç kuralı LKH'nin kendi listesine uygulanınca LKH'nin alfa = 0 girdilerini,
    # işaretleme yazıcısı da LKH'nin kendi işaretleriyle LKH'nin alfa değerlerini aynen vermelidir; uymayan girdi
    # sayısı kaydedilir ve raporda gösterilir.
    native_zero = present & (pop_alpha == 0)
    rule_mismatch = [int((A.tree_alpha(raw_pop[m], tree_keys[m])[1] != native_zero[m]).sum()) for m in range(W)]
    writer_mismatch = [int((K.mark_alpha(raw_pop[m], native_zero[m], rest=pop_alpha[m]) != pop_alpha[m])[present[m]].sum())
                       for m in range(W)]
    kept, zeros = {}, {}
    for arm, (cand_key, alpha_key, _) in ARM_FILES.items():
        cand, alpha = lists[cand_key], lists[alpha_key]
        kept[arm] = [A.kept_share(cand[m], alpha[m], tours[m]) for m in range(W)]
        zeros[arm] = float(((cand >= 0) & (alpha == 0)).sum() / (W * n))
    # Tur kenarlarının aday listelerindeki payı (işaretlemeden bağımsız; korunabilen payın üst sınırı).
    in_list = dict(pop=[A.kept_share(raw_pop[m], np.zeros_like(pop_alpha[m]), tours[m]) for m in range(W)],
                   beta=[A.kept_share(beta[m], np.zeros_like(beta_tree[m]), tours[m]) for m in range(W)])
    meta = dict(overhead_s={arm: float(v) for arm, v in overhead.items()},
                successor_build_s=float(successor_s), predict_wall_s=float(predict_s), beta_mix_s=float(beta_mix_s),
                beta_tree_s=float(beta_tree_s), model_threshold_s=float(threshold_s),
                pop_model_s=float(pop_model_s), beta_model_s=float(beta_model_s), beta_count_s=float(beta_count_s),
                model_predict_s=float(prediction["t_total"]),
                kept_share_by_seed=kept, kept_share={arm: float(np.mean(v)) for arm, v in kept.items()},
                zero_alpha_per_node=zeros,
                tour_edges_in_list={key: float(np.mean(v)) for key, v in in_list.items()},
                tree_rule_mismatch=rule_mismatch, mark_writer_mismatch=writer_mismatch,
                mark_count=int(MARK_COUNT), model_threshold=[float(t) for t in threshold],
                first_tour_edges_trusted_by_count=[float((own_count[m] >= MARK_COUNT).mean()) for m in range(W)],
                first_tour_edges_trusted_by_model=[float(K.trusted_by_probability(own_prob[m], threshold[m]).mean())
                                                   for m in range(W)],
                first_tour_edges_in_graph=[float((own_prob[m] > 0).mean()) for m in range(W)],
                first_tour_edges_trusted_by_both=[float(((own_count[m] >= MARK_COUNT)
                                                         & K.trusted_by_probability(own_prob[m], threshold[m])).mean())
                                                  for m in range(W)],
                pop_candidates_per_node=float(present.sum() / (W * n)),
                beta_top3_in_pop5=float(np.mean([(beta[m, :, :K_LEARNED, None] == raw_pop[m, :, None, :]).any(-1).mean()
                                                 for m in range(W)])))
    return lists, meta


def make_resumed_arm(inst, phase, phase_meta, cand, alpha, dad, overhead_s, arm_name):
    """Ortak ilk aşamadan sonra yeniden başlatma: aday dosyası (cand, alpha, dad), sıfır ceza, ilk tur.

    Bütçeden ilk aşamanın duvar süresi ve listenin hazırlık süresi düşülür; kalan süre LKH'nin TIME_LIMIT değeridir."""
    budget = inst["budget"]
    prefix = float(phase_meta["wall_s"]) + float(overhead_s)
    remaining = budget - prefix
    firsts = phase_meta["first_events"]
    if remaining <= 0.25:
        runs = [run_record(arm_name, seed, firsts[m], [firsts[m][0][1]], budget, search_start_s=None, wall_s=0.0,
                           returncode=None, timed_out=False, cost_check=None) for m, seed in enumerate(SEEDS)]
        return arm_record(runs, continuation_wall_s=0.0, prelude_s=float(prefix),
                          continuation_budget_s=max(0.0, float(remaining)), skipped_due_budget=True)
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(A.task_guided, [dict(common, budget=remaining, seed=SEEDS[m], cand=cand[m], alpha=alpha[m],
                                              dad=None if dad is None else dad[m], init_tour=phase["tours"][m],
                                              timeout=remaining + PAD_SECONDS) for m in range(W)])
    wall = time.time() - start
    runs = []
    for m, (seed, res) in enumerate(zip(SEEDS, raw)):
        assert exit_ok(res), f"{arm_name} / tohum {seed}: LKH çıkış kodu {res['returncode']}"
        offset = prefix + float(res["t_start"] - start)
        resumed = event_list(res, offset)
        if res["cost_check"] is not None and res["returncode"] == 0:
            resumed.append([int(res["cost_check"]), float(offset + res["wall"])])
        search_start = offset + float(res["pre_time"] or 0.0)
        runs.append(run_record(arm_name, seed, sorted(firsts[m] + resumed, key=lambda e: e[1]),
                               [firsts[m][0][1]] + [search_start + t for t in res["trial_times"]], budget,
                               search_start_s=search_start, wall_s=float(res["wall"]), returncode=res["returncode"],
                               timed_out=bool(res["timed_out"]),
                               cost_check=None if res["cost_check"] is None else int(res["cost_check"])))
    return arm_record(runs, continuation_wall_s=float(wall), prelude_s=float(prefix),
                      continuation_budget_s=float(remaining), skipped_due_budget=False)


def load_phase(stage_dir):
    meta_path, array_path = stage_dir / "phase1.json", stage_dir / "phase1.npz"
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None


def load_lists(stage_dir, name):
    meta_path, array_path = stage_dir / "candidates.json", OUTPUT_ROOT / "candidates" / (name + ".npz")
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None


def xy_hash(inst):
    return hashlib.sha256(np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest()


def run_one_instance(inst, ordinal):
    name = inst["name"]
    final_path = OUTPUT_ROOT / "instances" / (name + ".json")
    stage_dir = OUTPUT_ROOT / "stages" / name
    session_marker = stage_dir / "runtime_session_id.txt"
    if stage_dir.exists() and any(stage_dir.iterdir()):
        old_id = session_marker.read_text().strip() if session_marker.exists() else None
        if old_id != SESSION_ID:
            print("Yeni Colab makinesi: yarım kalan örnek baştan çalıştırılıyor:", name, flush=True)
            shutil.rmtree(stage_dir)
            (OUTPUT_ROOT / "candidates" / (name + ".npz")).unlink(missing_ok=True)
    stage_dir.mkdir(parents=True, exist_ok=True)
    session_marker.write_text(SESSION_ID)
    full_path = stage_dir / "pop_full.json"
    # Yeniden başlatılan kolların sırası örnek başına rastgeledir; kesintisiz kol, aile içi sıra numarası çiftse
    # önce, tekse sonra koşar (her ailede üç örnek önce, üç örnek sonra).
    control_order = [str(x) for x in np.random.default_rng(23000 + ordinal).permutation(RESTART_ARMS)]
    full_first = bool(inst["index"] % 2 == 0)

    def ensure_full():
        if not full_path.exists():
            print(name, "/ pop_full (kesintisiz)", flush=True)
            write_json_atomic(full_path, make_full_arm(inst))

    def ensure_controls():
        loaded = load_phase(stage_dir)
        if loaded is None:
            print(name, "/ ortak ilk denemeler", flush=True)
            phase, phase_meta = make_phase1(inst)
            write_npz_atomic(stage_dir / "phase1.npz", **phase)
            write_json_atomic(stage_dir / "phase1.json", phase_meta)
        else:
            phase, phase_meta = loaded
        loaded = load_lists(stage_dir, name)
        if loaded is None:
            print(name, "/ aday dosyaları", flush=True)
            lists, list_meta = make_candidate_lists(inst, phase)
            write_npz_atomic(OUTPUT_ROOT / "candidates" / (name + ".npz"), **lists)
            write_json_atomic(stage_dir / "candidates.json", list_meta)
        else:
            lists, list_meta = loaded
        for arm in control_order:
            path = stage_dir / (arm + ".json")
            if path.exists():
                continue
            print(name, "/", arm, flush=True)
            cand_key, alpha_key, with_dad = ARM_FILES[arm]
            write_json_atomic(path, make_resumed_arm(inst, phase, phase_meta, lists[cand_key], lists[alpha_key],
                                                     phase["pop_dad"] if with_dad else None,
                                                     list_meta["overhead_s"][arm], arm))
        return phase_meta, list_meta

    if full_first:
        ensure_full()
        phase_meta, list_meta = ensure_controls()
    else:
        phase_meta, list_meta = ensure_controls()
        ensure_full()
    arms = {arm: json.loads((stage_dir / (arm + ".json")).read_text()) for arm in ARM_NAMES}
    first_match = [int(arms["pop_full"]["runs"][m]["events"][0][0]) == int(phase_meta["first_events"][m][0][0])
                   for m in range(W)]
    data = dict(schema=1, complete=True, name=name, group=inst["group"], family=inst["family"],
                index=int(inst["index"]), ordinal=int(ordinal), n=len(inst["xy"]), budget_s=float(inst["budget"]),
                ew=inst["ew"], xy_sha256=xy_hash(inst), seeds=SEEDS, fractions=FRACTIONS,
                control_order=control_order, full_first=full_first, hardware=HARDWARE, session_id=SESSION_ID,
                code_sha256=CODE_SHA256, marking_sha256=MARKING_SHA256, phase1=phase_meta, lists=list_meta,
                first_trial_matches_uninterrupted=first_match, arms=arms,
                finished_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    write_json_atomic(final_path, data)
    print("TAMAM", name, "|", ", ".join(f"{arm}={arms[arm]['best_at_budget']}" for arm in ARM_NAMES), flush=True)
    return data

In [ ]:
def warmup():
    """Model ve LKH ısınması (ölçüme dahil değildir)."""
    xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
    succ = np.stack([np.roll(np.arange(100), -1)] * W, axis=1)
    M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=np.ones(W))
    first = A.task_first_trial(LKH, str(WORK_ROOT / "lkh_tmp"), SEEDS[0], xy, k=K_LIST, timeout=60)
    assert T.is_valid_tour(first["tour"], 100) and first["cand"].shape == (100, K_LIST) == first["alpha"].shape
    # Deney boyutunda bir tahmin: ilk örnek, GPU'nun tek seferlik hazırlık süresini ödemesin.
    big = T.to_int_coords(T.gen_instance(N_NODES, "uniform", np.random.default_rng(9091)))
    order = np.argsort(big[:, 0], kind="stable")
    succ_big = np.empty(N_NODES, dtype=np.int32)
    succ_big[order] = np.roll(order, -1)
    M.predict_tourfold(MODEL, big, K_GRAPH, DEVICE, ens_succ=np.stack([succ_big] * W, axis=1), ens_cost=np.ones(W))


warmup()

# Kodlama öz sınaması: bu makinede ve bu LKH ikilisiyle, sabit deneme sayılı (belirlenimci) koşular. LKH'nin kendi
# yazdığı aday dosyasıyla yapılan arama, yeniden yazılmış dört dosyayla yapılan aramalarla karşılaştırılır:
#   1) LKH'nin kendi alfa değerleri + baba alanı: her denemede aynı maliyet (dosyamız LKH'ninkiyle eşdeğer).
#   2) alfa = 0, 100, 200, ... + baba alanı: arama en erken ikinci denemede ayrılır (yeniden başlatma kuralı).
#   3) Baba alanı olmayan dosyalar (run v1-v8 böyle yazıldı): fark ilk denemede de başlayabilir.
# Ayrıca ağaç kuralının LKH'nin alfa = 0 girdilerini aynen ürettiği denetlenir.
SELFCHECK = A.encoding_selfcheck(LKH, str(WORK_ROOT / "lkh_tmp"),
                                 T.to_int_coords(T.gen_instance(2000, "uniform", np.random.default_rng(9092))),
                                 SEEDS[0], trials=25, k=K_LIST)
SELFCHECK["session_id"] = SESSION_ID
write_json_atomic(OUTPUT_ROOT / "summary" / "selfcheck.json", SELFCHECK)
first_diff = SELFCHECK["first_trial_with_a_different_cost"]
assert SELFCHECK["native_file_equals_lkh_file"], "LKH'nin kendi alfa ve baba değerleriyle yazılan dosya LKH'nin dosyasıyla aynı aramayı vermedi."
assert SELFCHECK["tree_rule_matches_lkh"], "Ağaç kuralı LKH'nin alfa = 0 girdilerini üretmedi."
assert first_diff["rank_dad"] is not None and first_diff["rank_dad"] >= 2, \
    "alfa = 0, 100, 200, ... kodlaması aramayı beklenen biçimde değiştirmedi; deneyin öncülü bu LKH ikilisinde geçerli değil."
print("Öz sınama tamam. Yeniden başlatmada korunabilen tur kenarı payı: LKH'nin kendi alfası "
      f"{SELFCHECK['kept_share_native']:.3f}, rank kodlaması {SELFCHECK['kept_share_rank']:.3f}. "
      f"Aramanın LKH'nin kendi dosyasından ayrıldığı ilk deneme: {first_diff}")

In [ ]:
def save_progress(last_instance=None, state="running"):
    finished = sorted(p.stem for p in (OUTPUT_ROOT / "instances").glob("*.json"))
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      dict(state=state, completed=len(finished), total=len(INSTANCES), finished=finished,
                           last_instance=last_instance,
                           updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))


save_progress(state="running")
session_start = time.time()
newly_finished = 0
for ordinal, inst in enumerate(INSTANCES):
    final_file = OUTPUT_ROOT / "instances" / (inst["name"] + ".json")
    if final_file.exists():
        record = json.loads(final_file.read_text())
        assert record.get("complete") and record.get("xy_sha256") == xy_hash(inst) \
            and set(record["arms"]) == set(ARM_NAMES), f"{inst['name']}: eski kayıt bu kurulumla uyuşmuyor"
        print("ATLA (tamam):", inst["name"], flush=True)
        continue
    if MAX_NEW_INSTANCES is not None and newly_finished >= MAX_NEW_INSTANCES:
        print("Bu oturum için MAX_NEW_INSTANCES sınırına ulaşıldı.")
        break
    try:
        run_one_instance(inst, ordinal)
        newly_finished += 1
        save_progress(last_instance=inst["name"])
        done_now = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
        per_instance_s = (time.time() - session_start) / newly_finished
        print(f"  {done_now}/{len(INSTANCES)} örnek bitti; örnek başına {per_instance_s / 60:.1f} dk, "
              f"kalan yaklaşık {(len(INSTANCES) - done_now) * per_instance_s / 3600:.1f} saat.", flush=True)
    except Exception as exc:
        failure = dict(instance=inst["name"], error=repr(exc), traceback=traceback.format_exc(),
                       utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + ".json"), failure)
        save_progress(last_instance=inst["name"], state="error")
        raise

done = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
save_progress(state="complete" if done == len(INSTANCES) else "partial")
print(f"Bu oturumda yeni {newly_finished}, toplam {done}/{len(INSTANCES)} örnek tamamlandı. "
      f"Süre: {(time.time() - session_start) / 3600:.2f} saat. Kayıt: {OUTPUT_ROOT}")

In [ ]:
# (anahtar, ilk kol, ikinci kol, açıklama). Yüzde fark = 100 x (ilk - ikinci) / ikinci; negatif: ilk kol daha kısa.
COMPARISONS = [
    ("M1", "beta_model", "beta_tree", "Model işaretlemesi / ağaç işaretlemesi (BetaTour listesi)"),
    ("M2", "beta_count", "beta_tree", "Sayım işaretlemesi / ağaç işaretlemesi (BetaTour listesi)"),
    ("M3", "pop_model", "pop_native", "Model işaretlemesi / ağaç işaretlemesi (POPMUSIC listesi)"),
    ("S1", "beta_model", "beta_count", "Model / sayım işaretlemesi (BetaTour listesi)"),
    ("S2", "beta_model", "pop_full", "BetaTour + model işaretlemesi / kesintisiz POPMUSIC"),
    ("S3", "beta_count", "pop_full", "BetaTour + sayım işaretlemesi / kesintisiz POPMUSIC"),
    ("S4", "beta_tree", "pop_full", "BetaTour, ağaç işaretlemesi / kesintisiz POPMUSIC (run v21'deki P2, yeni örnekler)"),
    ("S5", "pop_model", "pop_full", "POPMUSIC + model işaretlemesi / kesintisiz POPMUSIC"),
    ("S6", "pop_native", "pop_full", "Yeniden başlatmanın kendisi: LKH eşdeğeri dosya / kesintisiz"),
    ("S7", "beta_tree", "pop_native", "Listenin etkisi, ağaç işaretlemesi: BetaTour / POPMUSIC"),
    ("S8", "beta_model", "pop_model", "Listenin etkisi, model işaretlemesi: BetaTour / POPMUSIC"),
]
# Önceden belirlenen üç ana okuma (PROTOCOL_v23.md): bütçe sonunda, altı tohumun eşli ortalaması. Koşu keşif
# amaçlıdır; hiçbir satır doğrulayıcı bir test değildir.
MAIN = {"M1": "mean6", "M2": "mean6", "M3": "mean6"}
ENDPOINTS = ("best6", "mean6")


def pct(a, b):
    return 100.0 * (a - b) / b


def time_tag(f):
    return f"{int(round(100 * f)):03d}"


def load_finished():
    by_name = {x["name"]: x for x in INSTANCES}
    found = []
    for path in sorted((OUTPUT_ROOT / "instances").glob("*.json")):
        item = json.loads(path.read_text())
        if item.get("complete") and item["name"] in by_name:
            assert set(item["arms"]) == set(ARM_NAMES) and item["xy_sha256"] == xy_hash(by_name[item["name"]])
            assert all(len(item["arms"][arm]["runs"]) == W for arm in ARM_NAMES)
            found.append(item)
    return sorted(found, key=lambda r: r["ordinal"])


records = load_finished()
rows, seed_rows = [], []
for item in records:
    row = dict(instance=item["name"], group=item["group"], n=item["n"], budget_s=item["budget_s"],
               ordinal=item["ordinal"], full_first=item["full_first"], control_order=" ".join(item["control_order"]),
               first_match_all=all(item["first_trial_matches_uninterrupted"]), phase1_s=item["phase1"]["wall_s"],
               model_s=item["lists"]["model_predict_s"], beta_top3_in_pop5=item["lists"]["beta_top3_in_pop5"],
               pop_candidates_per_node=item["lists"]["pop_candidates_per_node"],
               tour_edges_in_pop=item["lists"]["tour_edges_in_list"]["pop"],
               tour_edges_in_beta=item["lists"]["tour_edges_in_list"]["beta"],
               model_threshold=float(np.mean(item["lists"]["model_threshold"])),
               first_tour_trusted_by_count=float(np.mean(item["lists"]["first_tour_edges_trusted_by_count"])),
               first_tour_trusted_by_model=float(np.mean(item["lists"]["first_tour_edges_trusted_by_model"])),
               first_tour_trusted_by_both=float(np.mean(item["lists"]["first_tour_edges_trusted_by_both"])),
               cpu_model=item["hardware"]["cpu_model"], gpu=item["hardware"]["gpu"], session=item["session_id"][:8])
    for arm in ARM_NAMES:
        data = item["arms"][arm]
        row[arm] = int(data["best_at_budget"])
        row[arm + "_mean6"] = float(data["mean_at_budget"])
        row[arm + "_trials"] = float(data["mean_trials_at_fraction"][-1])
        if arm in RESTART_ARMS:
            row[arm + "_kept"] = float(item["lists"]["kept_share"][arm])
            row[arm + "_zero_alpha"] = float(item["lists"]["zero_alpha_per_node"][arm])
            row[arm + "_prelude_s"] = float(data["prelude_s"])
            row[arm + "_overhead_s"] = float(item["lists"]["overhead_s"][arm])
        for run in data["runs"]:
            seed_rows.append(dict(instance=item["name"], group=item["group"], seed=run["seed"], arm=arm,
                                  cost=run["best_at_budget"], trials=run["trials_at_fraction"][-1]))
    for key, first, second, _ in COMPARISONS:
        for j, f in enumerate(FRACTIONS):
            a, b = item["arms"][first]["best_at_fraction"][j], item["arms"][second]["best_at_fraction"][j]
            row[f"{key}_{time_tag(f)}_best6"] = pct(a, b) if a is not None and b is not None else float("nan")
            sa = [r["best_at_fraction"][j] for r in item["arms"][first]["runs"]]
            sb = [r["best_at_fraction"][j] for r in item["arms"][second]["runs"]]
            complete = all(x is not None for x in sa + sb)
            row[f"{key}_{time_tag(f)}_mean6"] = (float(np.mean([pct(x, y) for x, y in zip(sa, sb)]))
                                                  if complete else float("nan"))
    rows.append(row)

per_instance = pd.DataFrame(rows)
per_seed = pd.DataFrame(seed_rows)
per_instance.to_csv(OUTPUT_ROOT / "summary" / "per_instance.csv", index=False)
per_seed.to_csv(OUTPUT_ROOT / "summary" / "per_seed.csv", index=False)

stat_rows, arm_rows = [], []
if len(per_instance):
    for group, frame in [("all", per_instance)] + list(per_instance.groupby("group", sort=True)):
        for ci, (key, first, second, label) in enumerate(COMPARISONS):
            for j, f in enumerate(FRACTIONS):
                for ei, endpoint in enumerate(ENDPOINTS):
                    values = frame[f"{key}_{time_tag(f)}_{endpoint}"].dropna().to_numpy()
                    if len(values):
                        stat_rows.append(dict(group=group, key=key, comparison=label, first=first, second=second,
                                              endpoint=endpoint, time_pct=int(round(100 * f)),
                                              main_endpoint=bool(MAIN.get(key) == endpoint),
                                              main=bool(group == "all" and f == FRACTIONS[-1]
                                                        and MAIN.get(key) == endpoint),
                                              **A.paired_stats(values, 20261123 + 1000 * ci + 10 * j + ei)))
        full_trials = frame["pop_full_trials"].mean()
        # En kısa tur: bir örnekte birden fazla kol eşitse her biri sayılır.
        best_cost = frame[ARM_NAMES].min(axis=1)
        best_mean = frame[[arm + "_mean6" for arm in ARM_NAMES]].min(axis=1)
        for arm in ARM_NAMES:
            restart = arm in RESTART_ARMS
            arm_rows.append(dict(group=group, arm=arm, instances=len(frame),
                                 shortest_best6=int((frame[arm] == best_cost).sum()),
                                 shortest_mean6=int((frame[arm + "_mean6"] == best_mean).sum()),
                                 mean_trials_at_budget=frame[arm + "_trials"].mean(),
                                 trials_vs_pop_full_pct=pct(frame[arm + "_trials"].mean(), full_trials),
                                 kept_share=frame[arm + "_kept"].mean() if restart else float("nan"),
                                 zero_alpha_per_node=frame[arm + "_zero_alpha"].mean() if restart else float("nan"),
                                 prelude_s=frame[arm + "_prelude_s"].mean() if restart else 0.0,
                                 list_overhead_s=frame[arm + "_overhead_s"].mean() if restart else 0.0))
stats = pd.DataFrame(stat_rows)
arm_table = pd.DataFrame(arm_rows)
stats.to_csv(OUTPUT_ROOT / "summary" / "paired_stats.csv", index=False)
arm_table.to_csv(OUTPUT_ROOT / "summary" / "arms.csv", index=False)

suite_complete = len(INSTANCES) > 0 and len(records) == len(INSTANCES)
if TEST_MODE:
    note = "TEKNİK ÖZ SINAMA: bu sonuç makale için kullanılmaz."
elif not suite_complete:
    note = f"ARA SONUÇ: {len(records)}/{len(INSTANCES)} örnek tamamlandı. Tam sonucu bekleyin."
else:
    note = f"{len(records)}/{len(INSTANCES)} örnek tamamlandı."


def reading(row):
    """Önceden yazılan karar kuralı: örnek düzeyinde kesin iki yönlü işaret testi p < 0,05 ise (12 örnekte eşitlik
    yoksa en az 10 örnek) ve yeni işaretleme daha çok örnekte kısaysa doğrulama koşusu hazırlanır."""
    if not suite_complete or TEST_MODE:
        return "undecided", "Koşu tamamlanmadan yorum yapılmaz."
    low, high = row["ci95_low_pct"], row["ci95_high_pct"]
    detail = (f"{row['win']}/{row['tie']}/{row['loss']} (ilk kol kısa/eşit/uzun), işaret testi p = {row['sign_p']:.4f} "
              f"(düzeltmesiz, keşif amaçlı), ortalama fark %{row['mean_delta_pct']:+.4f} (%95 GA {low:+.4f} ile {high:+.4f}), "
              f"ortanca %{row['median_delta_pct']:+.4f}")
    if row["sign_p"] < 0.05 and row["win"] > row["loss"]:
        return "confirm", "Yeni işaretleme daha kısa tur verdi: DOĞRULAMA KOŞUSU hazırlanır. " + detail
    if row["sign_p"] < 0.05 and row["loss"] > row["win"]:
        return "worse", "Yeni işaretleme daha uzun tur verdi: fikir bu biçimiyle bırakılır. " + detail
    return "no_difference", ("Karar kuralı sağlanmadı: doğrulama koşusu hazırlanmaz (bu, fark olmadığının kanıtı "
                             "değildir; güven aralığına bakın). " + detail)


columns = ["key", "comparison", "endpoint", "instances", "win", "tie", "loss", "mean_delta_pct", "median_delta_pct",
           "ci95_low_pct", "ci95_high_pct", "sign_p"]
lines = ["# BetaTour run v23: yeniden başlatma işaretlemesi (keşif koşusu)", "", note, "",
         "Bu koşu keşif amaçlıdır: p değerleri düzeltmesizdir ve hiçbir satır doğrulayıcı bir test değildir. Daha iyi",
         "görünen bir işaretleme, makalede yer almadan önce kendi protokolü olan ayrı bir koşuda sınanır.", "",
         "Yüzde fark = 100 x (ilk kol - ikinci kol) / ikinci kol; negatif değer ilk kolun daha kısa tur bulduğunu gösterir.",
         "win/tie/loss: ilk kolun daha kısa, eşit, daha uzun olduğu örnek sayısı. Güven aralığı örnek düzeyinde 5000",
         "bootstrap tekrarından, p değeri kesin iki yönlü işaret testinden gelir.",
         "best6: altı tohumun en iyisi. mean6: aynı tohumlu eşli yüzde farkların altı tohum ortalaması.",
         "Ana okumalar (PROTOCOL_v23.md, koşudan önce yazıldı): M1, M2 ve M3; mean6, bütçe sonunda. Karar kuralı:",
         "işaret testi p < 0,05 ve yeni işaretleme daha çok örnekte kısaysa o işaretleme için doğrulama koşusu hazırlanır.", ""]
mismatch = [r["name"] for r in records if not all(r["first_trial_matches_uninterrupted"])]
rule_mismatch = [r["name"] for r in records if any(r["lists"]["tree_rule_mismatch"])]
writer_mismatch = [r["name"] for r in records if any(r["lists"]["mark_writer_mismatch"])]
sessions = sorted({r["session_id"][:8] for r in records})
cpu_models = sorted({r["hardware"]["cpu_model"] for r in records})
if mismatch:
    lines += [f"UYARI: {len(mismatch)} örnekte kesintisiz kolun ilk denemesi ortak ilk aşamayla aynı maliyeti vermedi: "
              + ", ".join(mismatch), ""]
if rule_mismatch:
    lines += [f"UYARI: {len(rule_mismatch)} örnekte ağaç kuralı LKH'nin alfa = 0 girdilerini aynen üretmedi: "
              + ", ".join(rule_mismatch), ""]
if writer_mismatch:
    lines += [f"UYARI: {len(writer_mismatch)} örnekte işaretleme yazıcısı LKH'nin kendi alfa değerlerini aynen üretmedi: "
              + ", ".join(writer_mismatch), ""]
if len(cpu_models) > 1:
    lines += ["UYARI: örnekler birden fazla CPU modelinde koştu: " + "; ".join(cpu_models), ""]
if len(sessions) > 1:
    lines += [f"Not: koşu {len(sessions)} Colab oturumuna bölündü; her örneğin bütün kolları tek oturumda koştu.", ""]
MAIN_RESULT = {}
if len(stats):
    at_budget = stats[(stats["time_pct"] == 100)]
    whole = at_budget[at_budget["group"] == "all"]
    lines += ["## Ana okumalar", ""]
    for _, r in whole[whole["main"]].iterrows():
        decision, text = reading(r)
        MAIN_RESULT[r["key"]] = dict({k: (v.item() if hasattr(v, "item") else v) for k, v in r.items()},
                                     decision=decision)
        lines += [f"**{r['key']} ({r['endpoint']}): {r['comparison']}.** " + text, ""]
    lines += [f"## Bütçe sonunda bütün karşılaştırmalar ({len(records)} örnek birlikte)", "", "```",
              whole[columns].round(5).to_string(index=False), "```", "",
              "## Ailelere göre ana okumalar", "", "```",
              at_budget[(at_budget["group"] != "all") & at_budget["main_endpoint"]]
              [["group"] + columns].round(5).to_string(index=False), "```", "",
              "## Zaman içinde (bütçenin %25, %50, %75 ve %100'ünde; mean6)", "", "```",
              stats[(stats["group"] == "all") & (stats["endpoint"] == "mean6") & stats["key"].isin(list(MAIN))]
              [["key", "time_pct", "win", "tie", "loss", "mean_delta_pct", "ci95_low_pct", "ci95_high_pct", "sign_p"]]
              .round(5).to_string(index=False), "```", "",
              "## Kollar: en kısa tur, deneme sayısı ve yeniden başlatmada korunabilen kenar payı", "",
              "shortest_best6 / shortest_mean6: kolun, altı kol içinde en kısa turu (altı tohumun en iyisi / ortalaması)",
              "bulduğu örnek sayısı; eşitlikte her kol sayılır. kept_share: ilk denemenin turundaki kenarlardan, aday",
              "dosyasında alfası 0 olanların payı (LKH'nin yeniden başlatma kuralının koruyabildiği kenarlar); 1 - kept_share",
              "her denemede bozulmaya açık bırakılan paydır. mean_trials_at_budget: bütçe içinde tamamlanan deneme sayısı",
              "(tohum ortalaması). list_overhead_s: kolun bütçesinden düşülen hazırlık süresi (model, liste, işaretleme).",
              "", "```", arm_table.round(4).to_string(index=False), "```", ""]
lines += ["Ham tablolar: summary/per_instance.csv, per_seed.csv, paired_stats.csv (bütün zaman noktaları), arms.csv.",
          "Aday dosyaları candidates/, ilk aşama dizileri stages/<örnek>/phase1.npz içindedir."]
(OUTPUT_ROOT / "summary" / "RAPOR.md").write_text("\n".join(lines), encoding="utf-8")
write_json_atomic(OUTPUT_ROOT / "summary" / "main.json",
                  dict(complete=bool(suite_complete), instances=len(records), main=MAIN_RESULT,
                       first_trial_mismatch=mismatch, tree_rule_mismatch=rule_mismatch,
                       mark_writer_mismatch=writer_mismatch, sessions=sessions,
                       gpus=sorted({r["hardware"]["gpu"] for r in records})))
print("\n".join(lines))

In [ ]:
import zipfile


def make_zip(name, include):
    """ZIP önce yerel diskte kurulur, sonra sonuç klasörüne (Drive) kopyalanır."""
    local = WORK_ROOT / name
    count = 0
    with zipfile.ZipFile(local, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            relative = path.relative_to(OUTPUT_ROOT)
            if not path.is_file() or path.suffix in {".zip", ".tmp"} or path.name.endswith(".tmp.npz"):
                continue
            if include(relative):
                z.write(path, str(relative))
                count += 1
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
    digest = hashlib.sha256(local.read_bytes()).hexdigest()
    shutil.copy2(local, OUTPUT_ROOT / name)
    print(f"{name}: {count} dosya, {local.stat().st_size / 1e6:.1f} MB, SHA-256 {digest}")
    return local


# Sonuç ZIP'i: kayıtlar, tablolar, rapor ve çalışan kod (küçük; bunu gönderin).
# Denetim ZIP'i: aday dosyaları ve ilk aşama dizileri (büyük; Drive'da kalır).
result_zip = make_zip(EXPERIMENT_NAME + "_results.zip",
                      lambda rel: rel.parts[0] in {"instances", "summary", "code", "errors"}
                      or rel.name in {"manifest.json", "progress.json"})
audit_zip = make_zip(EXPERIMENT_NAME + "_audit_arrays.zip",
                     lambda rel: rel.suffix == ".npz" or rel.name in {"manifest.json", "phase1.json", "candidates.json"})
if SAVE_TO_DRIVE:
    from google.colab import files
    files.download(str(result_zip))      # tarayıcıya iner
print("İki ZIP de şu klasörde:", OUTPUT_ROOT)